# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uzairrateef-rgb/my-flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/uzairrateef-rgb/my-flyrank-internship.git
%cd my-flyrank-internship

%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query.")

Cloning into 'my-flyrank-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 128 (delta 38), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.87 MiB | 9.35 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/my-flyrank-internship
Connected. Ready to query.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
q = f"""
SELECT content_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""
con.sql(q).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────┬───────┐
│ content_hash_id │ report_date │   n   │
│     varchar     │    date     │ int64 │
├─────────────────┴─────────────┴───────┤
│                0 rows                 │
└───────────────────────────────────────┘



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"One row = one piece of content's daily performance record — a unique (content_hash_id, report_date) pair in fact_content_daily_performance, for month = 2026-03 (a mid-panel month, not the sealed final month 2026-06). Confirmed: 9,841,378 rows spanning 2026-03-01 to 2026-03-31, across 331,437 distinct content pieces."


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"Feature fields: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_engaged_sessions, scroll_events"
"Label/proxy field: none built into this table directly, I'll derive a decline/growth proxy by comparing early-month vs. late-month gsc_impressions for each content piece (built in Section 3/4)"
"Context fields: content_hash_id, client_hash_id, report_date, client_has_gsc, client_has_ga4"
"Excluded: fact_content_query_90d.parquet, it's query-level grain (salted query hashes), a different unit of analysis than content-day; mixing it in here would silently multiply rows per content piece. Also excluding ai_chatgpt/ai_perplexity/etc. columns for now too sparse and outside this lane's decision (content refresh, not AI-referral analysis)."


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
q = f"""
SELECT COUNT(*) AS total,
       SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┬───────────────┐
│  total  │ gsc_available │ ga4_available │
│  int64  │    int128     │    int128     │
├─────────┼───────────────┼───────────────┤
│ 9841378 │       3611061 │        413966 │
└─────────┴───────────────┴───────────────┘



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"Query 1 (grain): grouping by content_hash_id + report_date and checking for duplicates returned 0 rows confirms one row truly is one content piece on one day, with no accidental duplication."
"Query 2 (counts + span): 9,841,378 rows across 2026-03-01 to 2026-03-31, covering 331,437 distinct content pieces, this is the row count and date span claimed in Section 1"
"Query 3 (availability, IS TRUE): of all 9,841,378 rows, only 36.7% have gsc_data_available IS TRUE and just 4.2% have ga4_data_available IS TRUE. This means the majority of daily records in this month have no usable Search Console signal at all, and GA4 coverage is far sparser still any feature built from these fields needs to explicitly account for that, not silently treat missing as zero."


In [5]:
# Build a simple decline proxy at content level for March, using only GSC rows
gsc_rows = con.sql(f"""
    SELECT content_hash_id, report_date, gsc_impressions, gsc_clicks
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

# Split into first half vs second half of March
gsc_rows['half'] = gsc_rows['report_date'].apply(lambda d: 'first' if d.day <= 15 else 'second')
pivot = gsc_rows.groupby(['content_hash_id', 'half'])['gsc_impressions'].sum().unstack(fill_value=0)
pivot['declined'] = (pivot['second'] < pivot['first']).astype(int)

print(pivot['declined'].value_counts(normalize=True))

# THE TRAP: add a label-derived column on purpose
import numpy as np
pivot['pct_change'] = (pivot['second'] - pivot['first']) / pivot['first'].replace(0, np.nan)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_leaky = pivot[['first', 'second', 'pct_change']].fillna(0)
y = pivot['declined']
leaky_tree = DecisionTreeClassifier(max_depth=2).fit(X_leaky, y)
print("Leaky accuracy:", accuracy_score(y, leaky_tree.predict(X_leaky)))  # looks perfect — pct_change IS the label

# Remove the leak, keep the honest score
X_honest = pivot[['first']].fillna(0)  # only pre-decision info: first-half impressions
honest_tree = DecisionTreeClassifier(max_depth=2).fit(X_honest, y)
print("Honest accuracy:", accuracy_score(y, honest_tree.predict(X_honest)))

declined
0    0.62325
1    0.37675
Name: proportion, dtype: float64
Leaky accuracy: 1.0
Honest accuracy: 0.62325023481085


## 3b. The leakage trap

The trap: I built a simple label declined = 1 if a page's GSC impressions in the second half of March were lower than the first half (37.7% of pages declined, 62.3% did not). Then I deliberately added pct_change (the exact percentage change between first and second half) as a feature and trained a depth-2 tree on it.

Leaky accuracy: 1.0 perfect. That's not a good model, it's a broken experiment: pct_change isn't a signal that predicts the label, it's a different arithmetic expression of the exact same number the label was built from. The tree didn't learn anything it just found the shortcut back to its own answer.

After removing pct_change and training only on first (first-half impressions information that's actually knowable before the second half happens), accuracy dropped to 0.623. That's barely better than just guessing "no decline" every time (62.3% base rate) which is the honest, sobering truth: predicting decline from first-half impressions alone, with no other features, adds almost nothing. This is the real lesson from Notebook 2's leakage warning, now proven on the full warehouse: a perfect score is usually a sign something's wrong, not a sign of success, and the fix is always to ask "would I have known this before the outcome happened?"

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"This slice can't tell us why a page's traffic changed only whether it did, and even that is unreliable for most rows: 63.3% of rows lack GSC data and 95.8% lack GA4 data in this month alone. This isn't random missingness, it tracks which clients have GSC/GA4 connected (client_has_gsc, client_has_ga4), so any model trained only on rows with full data will systematically represent better-instrumented clients more than others. It's also an unbalanced panel across time: different clients' history starts at different points, so month-over-month comparisons for newer clients may be comparing partial data to partial data."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.